# Bengaluru Parking Violation Intelligence — Phase 3
### Commander-Facing Enforcement Dashboard & Patrol Route Planner

**Why a dashboard, and why built this way:**
Kaggle notebooks don't run a persistent backend server, so a Streamlit/Flask
app isn't directly hostable from here. Instead, Phase 3 builds a **fully
self-contained interactive HTML file** — all data embedded as JSON, all
interactivity in plain JavaScript (Chart.js + Leaflet), no server required.
This is the same approach used for the Phase 1/2 maps, scaled
up into a full multi-tab operations dashboard.




In [306]:
import pandas as pd
import numpy as np
import json
import os, glob
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import xgboost as xgb

print("✓ Libraries loaded")


✓ Libraries loaded


In [307]:
def find_file(preferred_path, filename):
    if os.path.exists(preferred_path):
        return preferred_path
    candidates = glob.glob(f"/kaggle/input/**/{filename}", recursive=True)
    if candidates:
        print(f"⚠ Auto-detected {filename} at: {candidates[0]}")
        return candidates[0]
    raise FileNotFoundError(
        f"{filename} not found. Run Phase 1 + Phase 2 first, or upload their outputs as a dataset."
    )

WORKDIR = "/kaggle/working"

hourly   = pd.read_csv(find_file(f"{WORKDIR}/eda_hourly.csv", "eda_hourly.csv"))
daily    = pd.read_csv(find_file(f"{WORKDIR}/eda_daily.csv", "eda_daily.csv"))
station  = pd.read_csv(find_file(f"{WORKDIR}/eda_station_load.csv", "eda_station_load.csv"))
veh      = pd.read_csv(find_file(f"{WORKDIR}/eda_vehicle_risk.csv", "eda_vehicle_risk.csv"))
hexdf    = pd.read_csv(find_file(f"{WORKDIR}/h3_hexagons_with_cii.csv", "h3_hexagons_with_cii.csv"))
priority = pd.read_csv(find_file(f"{WORKDIR}/enforcement_priority_ranking.csv", "enforcement_priority_ranking.csv"))
ts       = pd.read_csv(find_file(f"{WORKDIR}/daily_timeseries_features.csv", "daily_timeseries_features.csv"))
ts["date"] = pd.to_datetime(ts["date"])

model_path = find_file(f"{WORKDIR}/xgb_violation_forecast_model.json", "xgb_violation_forecast_model.json")
model = xgb.XGBRegressor()
model.load_model(model_path)

print(f"✓ Hourly        : {len(hourly)} rows")
print(f"✓ Daily         : {len(daily)} rows")
print(f"✓ Station load  : {len(station)} rows")
print(f"✓ Vehicle risk  : {len(veh)} rows")
print(f"✓ H3 hexagons   : {len(hexdf):,} zones")
print(f"✓ Priority list : {len(priority)} ranked enforcement zones")
print(f"✓ Time series   : {len(ts):,} rows")
print(f"✓ Forecast model loaded from {model_path}")


✓ Hourly        : 24 rows
✓ Daily         : 7 rows
✓ Station load  : 54 rows
✓ Vehicle risk  : 22 rows
✓ H3 hexagons   : 749 zones
✓ Priority list : 195 ranked enforcement zones
✓ Time series   : 7,150 rows
✓ Forecast model loaded from /kaggle/working/xgb_violation_forecast_model.json


## Live Model Inference — Predict Tomorrow's Volume

For every zone with recent history, we run the Phase 2 forecasting model on its
**latest** feature row to get a live "tomorrow" prediction. This is what gives
the dashboard its predictive edge — instead of just showing yesterday's
numbers, it tells a commander where violations are *expected* to spike next.


In [308]:
FEATURES = ["lag_1","lag_2","lag_3","lag_7","roll7_mean","roll7_std","dow","day_num"]

latest = ts.sort_values("date").groupby("h3_cell").tail(1).copy()
latest["predicted_tomorrow"] = np.clip(model.predict(latest[FEATURES]), 0, None).round(1)
latest["trend"] = np.where(
    latest["predicted_tomorrow"] > latest["roll7_mean"] * 1.1, "RISING",
    np.where(latest["predicted_tomorrow"] < latest["roll7_mean"] * 0.9, "FALLING", "STABLE")
)

forecast_lookup = latest.set_index("h3_cell")[["predicted_tomorrow","trend"]]
print(f"✓ Live forecasts generated for {len(latest)} zones\n")
latest[["h3_cell","date","count","roll7_mean","predicted_tomorrow","trend"]].sort_values(
    "predicted_tomorrow", ascending=False
).head(10)


✓ Live forecasts generated for 50 zones



,h3_cell,date,count,roll7_mean,predicted_tomorrow,trend
3145,8860145b55fffff,2024-04-07,220,147.285714,111.800003,FALLING
2716,8860145b43fffff,2024-04-07,156,70.571429,75.400002,STABLE
3431,8860145b5dfffff,2024-04-07,109,71.571429,72.500000,STABLE
2573,8860145b3bfffff,2024-04-07,47,60.714286,64.800003,STABLE
6577,8861892e91fffff,2024-04-07,90,58.571429,62.799999,STABLE
6863,8861892e9bfffff,2024-04-07,56,83.428571,54.700001,FALLING
3288,8860145b59fffff,2024-04-07,68,61.571429,52.000000,FALLING
6720,8861892e93fffff,2024-04-07,23,45.285714,44.500000,STABLE
2430,8860145b37fffff,2024-04-07,73,47.714286,38.200001,FALLING
1143,8860145b07fffff,2024-04-07,57,40.000000,33.799999,FALLING


## Greedy Patrol Route Optimiser

Turns the static priority ranking into an actual route a patrol unit could
drive: start at the #1 priority zone, then always move to the **nearest
unvisited zone in the top-30 priority pool**. This is a simple but effective
nearest-neighbour heuristic — not a full vehicle-routing solver, but more than
enough to demonstrate the concept and give a commander a usable starting plan.


In [309]:
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlmb = np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(p1)*np.cos(p2)*np.sin(dlmb/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

def build_patrol_route(zones_df, n_stops=8, candidate_pool=30, start_idx=0):
    """Nearest-neighbour route through the top-priority zones."""
    zones = zones_df.head(candidate_pool).reset_index(drop=True)
    remaining = list(range(len(zones)))
    remaining.remove(start_idx)
    route = [zones.iloc[start_idx]]
    total_km = 0.0
    current = start_idx
    while len(route) < n_stops and remaining:
        dists = [haversine_km(zones.iloc[current]["lat"], zones.iloc[current]["lon"],
                               zones.iloc[r]["lat"], zones.iloc[r]["lon"]) for r in remaining]
        nearest_i = int(np.argmin(dists))
        next_idx = remaining[nearest_i]
        total_km += dists[nearest_i]
        route.append(zones.iloc[next_idx])
        current = next_idx
        remaining.remove(next_idx)
    return pd.DataFrame(route).reset_index(drop=True), round(total_km, 1)

patrol_route, route_km = build_patrol_route(priority, n_stops=8)
print(f"✓ Suggested patrol route: {len(patrol_route)} stops, {route_km} km total\n")
patrol_route[["priority_rank","junction_name","total_risk","cii_tier"]]


✓ Suggested patrol route: 8 stops, 14.0 km total



,priority_rank,junction_name,total_risk,cii_tier
0,1,BTP051 - Safina Plaza Junction,24670,SEVERE
1,28,BTP141 - Palmgroove Junction,2050,MODERATE
2,12,BTP144 - Commando Hospital Junction,7371,MODERATE
3,13,"BTP148 - 17th Main, Doopanahalli Bus Stop",6442,MODERATE
4,16,"BTP148 - 17th Main, Doopanahalli Bus Stop",5484,MODERATE
5,10,Area-12.993N-77.667E,8005,MODERATE
6,11,Area-13.001N-77.679E,7086,MODERATE
7,21,Area-12.993N-77.692E,4497,MODERATE


## Build the Dashboard Data Payload

All data the dashboard needs gets serialised into a single JSON object, embedded
directly into the HTML. No API calls, no server — the dashboard is a static
file that works completely offline once generated.


In [310]:
def df_records(d, cols):
    return d[cols].replace({np.nan: None}).to_dict("records")

priority_enriched = priority.merge(
    forecast_lookup, left_on=None, right_index=True, how="left",
    left_index=False
) if False else priority.copy()

# Merge forecast info onto priority list via nearest H3 cell match (best-effort join)
# Priority rows already carry lat/lon; match each to its h3_cell via hexdf for the join key.
import h3
priority_enriched = priority.copy()
priority_enriched["h3_cell"] = priority_enriched.apply(
    lambda r: h3.latlng_to_cell(r["lat"], r["lon"], 8), axis=1
)
priority_enriched = priority_enriched.merge(forecast_lookup, on="h3_cell", how="left")
priority_enriched["predicted_tomorrow"] = priority_enriched["predicted_tomorrow"].fillna(0)
priority_enriched["trend"] = priority_enriched["trend"].fillna("NO DATA")

payload = {
    "meta": {
        "total_violations": int(hourly["count"].sum()),
        "total_zones": len(hexdf),
        "severe_high_zones": int(hexdf["cii_tier"].isin(["SEVERE","HIGH"]).sum()),
        "n_police_stations": len(station),
        "route_km": route_km,
    },
    "hourly": df_records(hourly, ["hour_ist","count"]),
    "daily": df_records(daily, ["day_of_week","count"]),
    "stations": df_records(station.head(10), ["police_station","total_risk","count"]),
    "vehicle": df_records(veh.head(8), ["best_vehicle_type","pct_of_total_risk","pct_of_count"]),
    "tier_counts": hexdf["cii_tier"].value_counts().reindex(
        ["SEVERE","HIGH","MODERATE","LOW"], fill_value=0
    ).to_dict(),
    "priority": df_records(priority_enriched.head(50),
        ["priority_rank","junction_name","n_violations","total_risk","cii","cii_tier",
         "dominant_type","dominant_vehicle","dominant_station","lat","lon",
         "predicted_tomorrow","trend"]),
    "route": df_records(patrol_route, ["priority_rank","junction_name","total_risk","lat","lon"]),
}

payload_json = json.dumps(payload)
print(f"✓ Dashboard payload built: {len(payload_json)/1024:.1f} KB")
print(f"  Zones in priority table: {len(payload['priority'])}")
print(f"  Patrol route stops     : {len(payload['route'])}")


✓ Dashboard payload built: 21.2 KB
  Zones in priority table: 50
  Patrol route stops     : 8


## Assemble the Interactive Dashboard

The dashboard has four tabs: **Overview** (KPIs + charts), **Live Map**
(Leaflet map with all ranked zones, colour-coded by CII tier), **Patrol
Route** (the optimised route with stop-by-stop detail), and **Enforcement
Ranking** (full sortable table of all 50 zones with live forecasts).

We use [Leaflet](https://leafletjs.com/) instead of folium here because the
map needs to react to the same embedded JSON payload as the rest of the
dashboard — a single HTML file, one data source, four synchronised views.


In [311]:
html_head = '''<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>Bengaluru Parking Intelligence — Command Dashboard</title>
<script src="https://cdnjs.cloudflare.com/ajax/libs/Chart.js/4.4.1/chart.umd.min.js"></script>
<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css"/>
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
<style>
*{box-sizing:border-box;margin:0;padding:0}
body{font-family:-apple-system,BlinkMacSystemFont,"Segoe UI",sans-serif;background:#f8fafc;color:#1e293b;font-size:14px}
header{background:#1e293b;color:#fff;padding:16px 28px;display:flex;align-items:center;gap:12px}
header h1{font-size:18px;font-weight:600}
header .sub{font-size:12px;color:#94a3b8;margin-left:auto}
nav{display:flex;gap:4px;padding:10px 28px;background:#fff;border-bottom:1px solid #e2e8f0}
nav button{font-size:13px;padding:8px 16px;border-radius:8px;border:none;background:transparent;cursor:pointer;color:#64748b;font-weight:500}
nav button.active{background:#1e293b;color:#fff}
.tab{display:none;padding:20px 28px}
.tab.active{display:block}
.grid-4{display:grid;grid-template-columns:repeat(4,1fr);gap:14px;margin-bottom:16px}
.kpi{background:#fff;border-radius:10px;padding:16px 18px;border:1px solid #e2e8f0}
.kpi .n{font-size:26px;font-weight:700;line-height:1.1}
.kpi .l{font-size:11px;color:#64748b;margin-top:4px}
.kpi.red .n{color:#dc2626}.kpi.orange .n{color:#ea580c}.kpi.blue .n{color:#2563eb}.kpi.green .n{color:#16a34a}
.charts{display:grid;grid-template-columns:1fr 1fr;gap:14px;margin-bottom:16px}
.card{background:#fff;border-radius:10px;padding:18px;border:1px solid #e2e8f0}
.card h2{font-size:12px;font-weight:600;color:#475569;text-transform:uppercase;letter-spacing:.04em;margin-bottom:12px}
canvas{max-height:230px}
#map{height:520px;border-radius:10px}
table{width:100%;border-collapse:collapse;font-size:12px}
th{background:#f1f5f9;padding:7px 10px;text-align:left;font-weight:600;color:#475569;cursor:pointer;user-select:none}
th:hover{background:#e2e8f0}
td{padding:7px 10px;border-bottom:1px solid #f1f5f9}
tr:hover td{background:#f8fafc}
.badge{display:inline-block;padding:2px 9px;border-radius:10px;font-size:11px;font-weight:600}
.badge.SEVERE{background:#fee2e2;color:#991b1b}
.badge.HIGH{background:#ffedd5;color:#9a3412}
.badge.MODERATE{background:#fef9c3;color:#854d0e}
.badge.LOW{background:#dcfce7;color:#166534}
.badge.RISING{background:#fee2e2;color:#991b1b}
.badge.FALLING{background:#dbeafe;color:#1e40af}
.badge.STABLE{background:#f1f5f9;color:#475569}
.route-stop{display:flex;align-items:center;gap:12px;padding:12px 14px;background:#fff;border:1px solid #e2e8f0;border-radius:10px;margin-bottom:8px}
.route-num{width:28px;height:28px;border-radius:50%;background:#1e293b;color:#fff;display:flex;align-items:center;justify-content:center;font-size:12px;font-weight:700;flex-shrink:0}
.route-info{flex:1}
.route-info b{font-size:13px}
.route-info .meta{font-size:11px;color:#64748b}
footer{text-align:center;padding:16px;color:#94a3b8;font-size:11px}
input#search{padding:7px 12px;border:1px solid #e2e8f0;border-radius:8px;font-size:13px;width:240px;margin-bottom:10px}
</style>
</head>
<body>
'''
print("✓ HTML head template built")


✓ HTML head template built


In [312]:
html_body = '''
<header>
  <span>🚗</span>
  <h1>Bengaluru Parking Violation Intelligence — Command Dashboard</h1>
  <span class="sub">Phase 3 · Live forecasts &amp; patrol planning</span>
</header>

<nav>
  <button class="active" onclick="showTab('overview', this)">Overview</button>
  <button onclick="showTab('map', this)">Live Map</button>
  <button onclick="showTab('route', this)">Patrol Route</button>
  <button onclick="showTab('ranking', this)">Enforcement Ranking</button>
</nav>

<div id="tab-overview" class="tab active">
  <div class="grid-4">
    <div class="kpi red"><div class="n" id="kpi-total">-</div><div class="l">Total violations analysed</div></div>
    <div class="kpi orange"><div class="n" id="kpi-zones">-</div><div class="l">SEVERE + HIGH risk zones</div></div>
    <div class="kpi blue"><div class="n" id="kpi-stations">-</div><div class="l">Police stations covered</div></div>
    <div class="kpi green"><div class="n" id="kpi-route">-</div><div class="l">Optimised patrol route (km)</div></div>
  </div>
  <div class="charts">
    <div class="card"><h2>Enforcement activity by hour (IST)</h2><canvas id="chart-hourly"></canvas></div>
    <div class="card"><h2>Violations by day of week</h2><canvas id="chart-daily"></canvas></div>
    <div class="card"><h2>Top 10 police stations by risk score</h2><canvas id="chart-stations"></canvas></div>
    <div class="card"><h2>Congestion Impact tier distribution</h2><canvas id="chart-tiers"></canvas></div>
  </div>
</div>

<div id="tab-map" class="tab">
  <div class="card"><h2>Live enforcement priority map</h2><div id="map"></div></div>
</div>

<div id="tab-route" class="tab">
  <div class="card">
    <h2>Suggested patrol route — nearest-neighbour optimised</h2>
    <p style="font-size:12px;color:#64748b;margin-bottom:14px">
      Starting at the #1 priority zone, this route greedily visits the nearest unvisited
      high-priority zone at each step. Total distance shown in the Overview KPI.
    </p>
    <div id="route-list"></div>
  </div>
</div>

<div id="tab-ranking" class="tab">
  <div class="card">
    <h2>Full enforcement priority ranking (click headers to sort)</h2>
    <input type="text" id="search" placeholder="Filter by junction or station...">
    <table id="ranking-table">
      <thead>
        <tr>
          <th onclick="sortTable(0)">Rank</th>
          <th onclick="sortTable(1)">Junction</th>
          <th onclick="sortTable(2)">Violations</th>
          <th onclick="sortTable(3)">Risk score</th>
          <th onclick="sortTable(4)">CII</th>
          <th onclick="sortTable(5)">Tier</th>
          <th onclick="sortTable(6)">Predicted tomorrow</th>
          <th onclick="sortTable(7)">Trend</th>
          <th onclick="sortTable(8)">Station</th>
        </tr>
      </thead>
      <tbody id="ranking-body"></tbody>
    </table>
  </div>
</div>

<footer>Phase 3 dashboard · Generated from Phase 1 + Phase 2 model outputs · All data embedded, works offline</footer>
'''
print("✓ HTML body template built")


✓ HTML body template built


In [313]:
js_code = '''
<script>
const DATA = ''' + payload_json + ''';

function showTab(id, btn) {
  document.querySelectorAll(".tab").forEach(t => t.classList.remove("active"));
  document.querySelectorAll("nav button").forEach(b => b.classList.remove("active"));
  document.getElementById("tab-" + id).classList.add("active");
  btn.classList.add("active");
  if (id === "map" && !window._mapInit) { initMap(); window._mapInit = true; }
}

// ---- KPIs ----
document.addEventListener("DOMContentLoaded", () => {
  document.getElementById("kpi-total").textContent = DATA.meta.total_violations.toLocaleString();
  document.getElementById("kpi-zones").textContent = DATA.meta.severe_high_zones;
  document.getElementById("kpi-stations").textContent = DATA.meta.n_police_stations;
  document.getElementById("kpi-route").textContent = DATA.meta.route_km;
  buildCharts();
  buildRoute();
  buildTable();
});

// ---- Charts ----
function buildCharts() {
  const C = {red:"#ef4444",orange:"#f97316",amber:"#f59e0b",blue:"#3b82f6",teal:"#14b8a6",green:"#22c55e"};

  new Chart(document.getElementById("chart-hourly"), {
    type: "bar",
    data: {
      labels: DATA.hourly.map(d => String(d.hour_ist).padStart(2,"0") + ":00"),
      datasets: [{data: DATA.hourly.map(d => d.count), backgroundColor: C.blue+"bb", borderRadius: 4}]
    },
    options: {plugins:{legend:{display:false}}, scales:{x:{ticks:{font:{size:9}}}}}
  });

  new Chart(document.getElementById("chart-daily"), {
    type: "bar",
    data: {
      labels: DATA.daily.map(d => d.day_of_week),
      datasets: [{data: DATA.daily.map(d => d.count), backgroundColor: C.teal+"bb", borderRadius: 4}]
    },
    options: {plugins:{legend:{display:false}}}
  });

  new Chart(document.getElementById("chart-stations"), {
    type: "bar",
    data: {
      labels: DATA.stations.map(d => d.police_station),
      datasets: [{data: DATA.stations.map(d => d.total_risk), backgroundColor: C.orange+"bb", borderRadius: 4}]
    },
    options: {indexAxis:"y", plugins:{legend:{display:false}}, scales:{y:{ticks:{font:{size:10}}}}}
  });

  const tiers = DATA.tier_counts;
  new Chart(document.getElementById("chart-tiers"), {
    type: "doughnut",
    data: {
      labels: Object.keys(tiers),
      datasets: [{data: Object.values(tiers), backgroundColor: [C.red,C.orange,C.amber,C.green]}]
    },
    options: {plugins:{legend:{position:"right",labels:{font:{size:11}}}}}
  });
}

// ---- Leaflet map ----
function initMap() {
  const map = L.map("map").setView([12.9716, 77.5946], 12);
  L.tileLayer("https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}{r}.png", {
    attribution: "&copy; OpenStreetMap &copy; CARTO"
  }).addTo(map);

  const tierColor = {SEVERE:"#dc2626", HIGH:"#ea580c", MODERATE:"#d97706", LOW:"#22c55e"};

  DATA.priority.forEach(z => {
    const color = tierColor[z.cii_tier] || "#94a3b8";
    const radius = 6 + Math.min(z.priority_rank, 30) * -0.15;
    const marker = L.circleMarker([z.lat, z.lon], {
      radius: Math.max(radius, 4), color: color, fillColor: color, fillOpacity: 0.7, weight: 1
    }).addTo(map);
    marker.bindPopup(
      "<b>#" + z.priority_rank + " — " + z.junction_name + "</b><br>" +
      "Violations: <b>" + z.n_violations.toLocaleString() + "</b><br>" +
      "Risk score: <b>" + z.total_risk.toLocaleString() + "</b><br>" +
      "CII: <b>" + z.cii + "</b> (" + z.cii_tier + ")<br>" +
      "Predicted tomorrow: <b>" + z.predicted_tomorrow + "</b> (" + z.trend + ")<br>" +
      "Top violation: " + z.dominant_type
    );
  });
}

// ---- Patrol route ----
function buildRoute() {
  const container = document.getElementById("route-list");
  container.innerHTML = DATA.route.map((s, i) => (
    "<div class=\'route-stop\'>" +
      "<div class=\'route-num\'>" + (i+1) + "</div>" +
      "<div class=\'route-info\'>" +
        "<b>" + s.junction_name + "</b><br>" +
        "<span class=\'meta\'>Priority rank #" + s.priority_rank + " · Risk score " + s.total_risk.toLocaleString() + "</span>" +
      "</div>" +
    "</div>"
  )).join("");
}

// ---- Ranking table ----
function buildTable() {
  renderRows(DATA.priority);
  document.getElementById("search").addEventListener("input", e => {
    const q = e.target.value.toLowerCase();
    const filtered = DATA.priority.filter(z =>
      z.junction_name.toLowerCase().includes(q) || z.dominant_station.toLowerCase().includes(q)
    );
    renderRows(filtered);
  });
}

function renderRows(rows) {
  const tierBadge = t => "<span class=\'badge " + t + "\'>" + t + "</span>";
  const trendBadge = t => "<span class=\'badge " + t + "\'>" + t + "</span>";
  document.getElementById("ranking-body").innerHTML = rows.map(z => (
    "<tr>" +
      "<td>" + z.priority_rank + "</td>" +
      "<td>" + z.junction_name + "</td>" +
      "<td>" + z.n_violations.toLocaleString() + "</td>" +
      "<td><b>" + z.total_risk.toLocaleString() + "</b></td>" +
      "<td>" + z.cii + "</td>" +
      "<td>" + tierBadge(z.cii_tier) + "</td>" +
      "<td>" + z.predicted_tomorrow + "</td>" +
      "<td>" + trendBadge(z.trend) + "</td>" +
      "<td>" + z.dominant_station + "</td>" +
    "</tr>"
  )).join("");
}

let sortDir = {};
function sortTable(col) {
  sortDir[col] = !sortDir[col];
  const keys = ["priority_rank","junction_name","n_violations","total_risk","cii","cii_tier","predicted_tomorrow","trend","dominant_station"];
  const key = keys[col];
  const sorted = [...DATA.priority].sort((a,b) => {
    if (typeof a[key] === "string") return sortDir[col] ? a[key].localeCompare(b[key]) : b[key].localeCompare(a[key]);
    return sortDir[col] ? a[key]-b[key] : b[key]-a[key];
  });
  renderRows(sorted);
}
</script>
</body>
</html>
'''
print("✓ JavaScript logic built")


✓ JavaScript logic built


In [314]:
full_html = html_head + html_body + js_code
print(f"✓ Full dashboard assembled: {len(full_html)/1024:.1f} KB total")


✓ Full dashboard assembled: 32.8 KB total


## Save & Render the Dashboard

In [315]:
from pathlib import Path
from IPython.display import HTML, IFrame

OUTPUT_DIR = Path("/kaggle/working")
dashboard_path = OUTPUT_DIR / "phase3_command_dashboard.html"
dashboard_path.write_text(full_html, encoding="utf-8")

print(f"✓ Dashboard saved to {dashboard_path}  ({dashboard_path.stat().st_size // 1024} KB)")
print("  Open this file directly in any browser — fully offline, no server needed.")


✓ Dashboard saved to /kaggle/working/phase3_command_dashboard.html  (32 KB)
  Open this file directly in any browser — fully offline, no server needed.


In [316]:
from IPython.display import HTML

html_content = dashboard_path.read_text(encoding="utf-8")
escaped = html_content.replace('"', '&quot;')

display(HTML(f'<iframe srcdoc="{escaped}" width="100%" height="800" style="border:1px solid #e2e8f0;border-radius:10px;"></iframe>'))

Rank,Junction,Violations,Risk score,CII,Tier,Predicted tomorrow,Trend,Station


## Summary

Phase 3 complete. The dashboard combines everything built across all three
notebooks into one operational view:

- **Overview tab** — city-wide KPIs and the same EDA charts from Phase 1, now live
- **Live Map tab** — every ranked zone from Phase 2's DBSCAN clustering, colour-coded by Congestion Impact Index, with live next-day forecasts in each popup
- **Patrol Route tab** — an actual drivable route through the highest-priority zones
- **Enforcement Ranking tab** — the full sortable, searchable priority table a commander would use to brief field units


In [317]:
print("="*55)
print("PHASE 3 COMPLETE")
print("="*55)
print(f'''
Dashboard file     : {dashboard_path}
File size          : {dashboard_path.stat().st_size // 1024} KB
Zones on live map  : {len(payload["priority"])}
Patrol route stops : {len(payload["route"])} ({route_km} km)
Tabs               : Overview, Live Map, Patrol Route, Enforcement Ranking

This is the final artifact for the hackathon demo — open phase3_command_dashboard.html
in any browser. No installation, no server, fully self-contained.
''')


PHASE 3 COMPLETE

Dashboard file     : /kaggle/working/phase3_command_dashboard.html
File size          : 32 KB
Zones on live map  : 50
Patrol route stops : 8 (14.0 km)
Tabs               : Overview, Live Map, Patrol Route, Enforcement Ranking

This is the final artifact for the hackathon demo — open phase3_command_dashboard.html
in any browser. No installation, no server, fully self-contained.

